# 🌤️ ContextAware: A Hybrid Fuzzy-Cognitive Dynamic Recommendation Framework
### Ambient Micro-Contextual & Culturally-Grounded Suggestion Engine
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mostafa-ja/contextAware/blob/main/context_aware_engine.ipynb)
![Python 3.8+](https://img.shields.io/badge/Python-3.8%2B-blue.svg)
![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)
![Architecture: Zero-Dependency Core](https://img.shields.io/badge/Core-Zero--Dependency-orange.svg)

---

## 📌 Executive Summary & Academic Scope
In ubiquitous computing and ambient intelligence, recommender systems frequently suffer from **cold-start constraints**, **rigid threshold brittleness**, and **contextual blindness**. Standard collaborative filtering methods assume static user-item interaction histories that fail to adapt when physical environmental conditions (temperature, storm warnings, circadian phases) dynamically govern human physical feasibility and emotional willingness.

**ContextAware** is a modular, transparent, and computationally lightweight **Context-Aware Recommender System (CARS)** designed to deliver instant, ambient lifestyle recommendations (Food, Activities, Media, Clothing, and Mood support) grounded in **fuzzy logic perception modeling**, **circadian harmonic alignment**, **Persian cultural temporalities**, and **multi-objective bundle synthesis**.

### 🔬 Core Theoretical Pillars:
1. **Continuous Fuzzy Membership Sets**: Converts crisp sensory readings into human-like multi-valued perception degrees ($\mu \in [0, 1]$) across temperature, humidity, wind, and time.
2. **Cognitive State Inference**: Infers high-order latent psychological variables (mood, social readiness, energy level, indoor vs. outdoor viability) via bounded t-conorm operators with diminishing returns.
3. **Persian Cultural & Temporal Grounding**: Encodes Solar Hijri (Jalali) seasonal transitions and culturally distinctive traditions (*Shab-e Yalda*, *Norooz*, *Sizdah Bedar*, mourning days).
4. **Feasibility Veto & Cosine-Projected Scoring**: Combines hard safety pruning ($p_{ik} \le -9.0$) with multi-group weighted dot products.
5. **Non-Linear Freshness Kinetics**: Eliminates recommendation fatigue and habituation via exponential decay and subcategory-specific recovery curves.
6. **Smart Bundle Affinity Graph**: Couples primary "Hero" recommendations with cohesive, diverse "Sidekicks" sampled via Boltzmann exploration.


---
## 1. ⚙️ Environment Setup & Auto-Configuration
This notebook runs seamlessly both **locally** and directly on **Google Colab**.
If executed on Colab, it automatically clones the repository and loads the curated 1,400+ Persian suggestion knowledge base.


In [ ]:
import os
import sys
import json
import math
import random
import datetime
import urllib.request
from typing import Dict, List, Any, Tuple, Optional, Set

# Visualizations
import matplotlib.pyplot as plt
import numpy as np

# Set high-resolution plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 120

# Google Colab automatic repository setup
try:
    import google.colab
    IN_COLAB = True
    print("🚀 Running in Google Colab environment.")
    if not os.path.exists("dataset"):
        print("📥 Cloning contextAware repository to load dataset...")
        os.system("git clone https://github.com/mostafa-ja/contextAware.git")
        import shutil
        if os.path.exists("contextAware/dataset"):
            shutil.copytree("contextAware/dataset", "./dataset", dirs_exist_ok=True)
        if os.path.exists("contextAware/src"):
            shutil.copytree("contextAware/src", "./src", dirs_exist_ok=True)
        print("✅ Dataset successfully mounted!")
except ImportError:
    IN_COLAB = False
    print("💻 Running in local Python environment.")

# Add local path to sys.path
sys.path.insert(0, ".")
print(f"✅ Ready. Python version: {sys.version.split()[0]}")


---
## 2. 🧠 Mathematical Formulation: Continuous Fuzzy Membership Functions

Crisp categorization (e.g., classifying $14.9^\circ\text{C}$ as "Cold" and $15.0^\circ\text{C}$ as "Warm") creates severe artificial discontinuities at decision boundaries. Humans perceive sensory conditions as overlapping fuzzy concepts. 

ContextAware utilizes parameterized **Triangular**, **Gaussian**, and **Trapezoidal** membership functions:

### 📐 Formulations:
1. **Triangular Membership**:
   $$\mu_{\text{tri}}(x; c, w) = \max\left(0, 1 - \frac{|x - c|}{w}\right)$$

2. **Gaussian Membership**:
   $$\mu_{\text{gauss}}(x; c, \sigma) = \exp\left(-\frac{1}{2} \left(\frac{x - c}{\sigma}\right)^2\right)$$

3. **Trapezoidal Membership**:
   $$\mu_{\text{trap}}(x; a, b, c, d) = \begin{cases}
   0 & x \le a \text{ or } x \ge d \\
   \frac{x - a}{b - a} & a < x < b \\
   1 & b \le x \le c \\
   \frac{d - x}{d - c} & c < x < d
   \end{cases}$$

Let's implement and visualize these continuous perception curves across the entire operational domain.


In [ ]:
def fuzzy_triangular(value: float, center: float, width: float) -> float:
    """Computes triangular membership degree in [0.0, 1.0]."""
    distance = abs(value - center)
    return max(0.0, min(1.0, 1.0 - distance / width))

def fuzzy_gaussian(value: float, center: float, sigma: float) -> float:
    """Computes Gaussian membership degree in [0.0, 1.0]."""
    if sigma <= 0:
        return 1.0 if value == center else 0.0
    val = math.exp(-0.5 * ((value - center) / sigma) ** 2)
    return 0.0 if val < 1e-5 else val

def fuzzy_trapezoidal(value: float, a: float, b: float, c: float, d: float) -> float:
    """Computes trapezoidal membership degree in [0.0, 1.0]."""
    if value <= a or value >= d:
        return 0.0
    if b <= value <= c:
        return 1.0
    if a < value < b:
        return (value - a) / (b - a) if b > a else 1.0
    if c < value < d:
        return (d - value) / (d - c) if d > c else 1.0
    return 0.0

# Fuzzy Set Hyperparameters for Normalized Input Signals
FUZZY_CONFIG = {
    # Temperature: norm = (T_effective + 10) / 50 in [-10°C, 40°C]
    "temp_extreme_cold": {"type": "gaussian",   "center": 0.05, "sigma": 0.055},
    "temp_cold":         {"type": "triangular", "center": 0.20, "width": 0.20},
    "temp_cool":         {"type": "trapezoidal","a": 0.28, "b": 0.38, "c": 0.52, "d": 0.64},
    "temp_warm":         {"type": "triangular", "center": 0.70, "width": 0.22},
    "temp_hot":          {"type": "gaussian",   "center": 0.90, "sigma": 0.085},

    # Humidity: norm = RH% / 100
    "humidity_very_dry":    {"type": "triangular", "center": 0.10, "width": 0.22},
    "humidity_dry":         {"type": "triangular", "center": 0.25, "width": 0.22},
    "humidity_comfortable": {"type": "trapezoidal","a": 0.37, "b": 0.43, "c": 0.55, "d": 0.63},
    "humidity_humid":       {"type": "triangular", "center": 0.70, "width": 0.22},
    "humidity_very_humid":  {"type": "gaussian",   "center": 0.88, "sigma": 0.10},

    # Circadian Hours: hour in [0.0, 24.0)
    "time_late_night":    {"type": "triangular", "center": 2.0,  "width": 5.0},
    "time_early_morning": {"type": "triangular", "center": 6.0,  "width": 2.0},
    "time_morning":       {"type": "triangular", "center": 9.25, "width": 2.25},
    "time_afternoon":     {"type": "triangular", "center": 15.0, "width": 2.5},
    "time_evening":       {"type": "triangular", "center": 19.0, "width": 1.5},
    "time_night":         {"type": "triangular", "center": 21.5, "width": 2.5},
}

def apply_fuzzy(feature_name: str, value: float) -> float:
    cfg = FUZZY_CONFIG.get(feature_name)
    if not cfg:
        return 0.0
    t = cfg["type"]
    if t == "triangular":
        return fuzzy_triangular(value, cfg["center"], cfg["width"])
    if t == "gaussian":
        return fuzzy_gaussian(value, cfg["center"], cfg["sigma"])
    if t == "trapezoidal":
        return fuzzy_trapezoidal(value, cfg["a"], cfg["b"], cfg["c"], cfg["d"])
    return 0.0


### 📊 Visualizing Fuzzy Perception Sets
Below, we plot how temperature, humidity, and time are smoothly transformed into continuous membership activations.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# 1. Temperature (-10°C to 40°C)
temp_c = np.linspace(-10, 40, 300)
temp_norm = (temp_c + 10.0) / 50.0

for key, color in zip(["temp_extreme_cold", "temp_cold", "temp_cool", "temp_warm", "temp_hot"],
                       ["#1f77b4", "#2ca02c", "#17becf", "#ff7f0e", "#d62728"]):
    y = [apply_fuzzy(key, n) for n in temp_norm]
    axes[0].plot(temp_c, y, label=key.replace("temp_", "").capitalize(), color=color, lw=2)

axes[0].set_title("Temperature Perception Sets", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Effective Temperature (°C)")
axes[0].set_ylabel("Membership Degree μ(x)")
axes[0].legend(loc="upper right")
axes[0].set_ylim(-0.05, 1.05)

# 2. Humidity (0% to 100%)
rh = np.linspace(0, 100, 300)
rh_norm = rh / 100.0

for key, color in zip(["humidity_very_dry", "humidity_dry", "humidity_comfortable", "humidity_humid", "humidity_very_humid"],
                       ["#8c564b", "#e377c2", "#2ca02c", "#1f77b4", "#9467bd"]):
    y = [apply_fuzzy(key, n) for n in rh_norm]
    axes[1].plot(rh, y, label=key.replace("humidity_", "").capitalize(), color=color, lw=2)

axes[1].set_title("Relative Humidity Sets", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Relative Humidity (%)")
axes[1].legend(loc="upper right")
axes[1].set_ylim(-0.05, 1.05)

# 3. Circadian Time (00:00 to 24:00)
hours = np.linspace(0, 24, 300)

for key, color in zip(["time_early_morning", "time_morning", "time_afternoon", "time_evening", "time_night", "time_late_night"],
                       ["#e7ba52", "#2ca02c", "#ff7f0e", "#9467bd", "#1f77b4", "#393b79"]):
    y = [apply_fuzzy(key, h) for h in hours]
    axes[2].plot(hours, y, label=key.replace("time_", "").capitalize(), color=color, lw=2)

axes[2].set_title("Circadian Time Windows", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Local Time (Decimal Hours)")
axes[2].legend(loc="upper right")
axes[2].set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()


---
## 3. 📐 The 65-Dimensional Feature Space & Cognitive Inference Engine

The Master Context Vector $\mathbf{c} \in [0, 1]^{65}$ spans 11 structured semantic domains:

$$\mathbf{c} = [\mathbf{c}_{\text{temp}}, \mathbf{c}_{\text{weather}}, \mathbf{c}_{\text{humidity}}, \mathbf{c}_{\text{wind}}, \mathbf{c}_{\text{time}}, \mathbf{c}_{\text{day}}, \mathbf{c}_{\text{season}}, \mathbf{c}_{\text{events}}, \mathbf{c}_{\text{mood}}, \mathbf{c}_{\text{social}}, \mathbf{c}_{\text{location}}, \mathbf{c}_{\text{energy}}]$$

### 🧠 Diminishing Returns Operator ($\oplus$):
Latent emotional and social states are not directly observed by sensors. When atmospheric precipitation occurs, it modifies the baseline mood through a **bounded t-conorm operator with diminishing returns**, preventing artificial saturation at $1.0$:

$$x \oplus \Delta = x + \Delta \cdot (1 - x) \cdot \alpha$$

where $\alpha = 0.70$ is the saturation damping parameter.


In [ ]:
# Define Feature Group Weights (Cognitive Priority in Ambient Matching)
GROUP_WEIGHTS = {
    "temp": 1.0,        # Critical physical thermal comfort
    "weather": 0.9,     # Feasibility constraint (rain, snow, storm)
    "social": 0.6,      # Contextual social dynamics
    "mood": 0.6,        # Emotional alignment
    "time": 1.0,        # Circadian / temporal constraint
    "location": 1.0,    # Physical location viability (indoor vs outdoor)
    "events": 0.8,      # Cultural and festive relevance
    "humidity": 0.6,    # Secondary comfort
    "wind": 0.6,        # Secondary comfort & outdoor feasibility
    "season": 0.5,      # Broad seasonal context
    "energy": 0.5,      # Personal fatigue/energy constraint
}

# Master 65-D Feature Taxonomy
FEATURE_NAMES = [
    # Temperature (5)
    "temp_extreme_cold", "temp_cold", "temp_cool", "temp_warm", "temp_hot",
    # Weather (10)
    "weather_clear", "weather_partly_cloudy", "weather_cloudy", "weather_fog",
    "weather_drizzle", "weather_rain", "weather_rain_shower", "weather_snow",
    "weather_snow_shower", "weather_thunderstorm",
    # Humidity (5)
    "humidity_very_dry", "humidity_dry", "humidity_comfortable", "humidity_humid", "humidity_very_humid",
    # Wind (5)
    "wind_calm", "wind_breeze", "wind_windy", "wind_strong", "wind_storm",
    # Time (6)
    "time_late_night", "time_early_morning", "time_morning", "time_afternoon", "time_evening", "time_night",
    # Day Type (4)
    "day_weekend", "day_holiday", "day_holiday_eve", "day_workday",
    # Season (4)
    "season_spring", "season_summer", "season_autumn", "season_winter",
    # Events (4)
    "romantic_event", "national_festival", "national_mourning", "cultural_tradition",
    # Social (5)
    "social_solo", "social_couple", "social_family", "social_friends", "social_group",
    # Mood (9)
    "mood_calm", "mood_energetic", "mood_happy", "mood_sad", "mood_thoughtful",
    "mood_romantic", "mood_nostalgic", "mood_stressed", "mood_relaxed",
    # Location (3)
    "location_indoor", "location_outdoor", "location_home",
    # Energy (5)
    "energy_very_low", "energy_low", "energy_medium", "energy_high", "energy_very_high"
]

FEATURE_GROUP = {}
for feat in FEATURE_NAMES:
    for group in GROUP_WEIGHTS.keys():
        if feat.startswith(group + "_"):
            FEATURE_GROUP[feat] = group
            break

print(f"✅ Feature Space Configured: {len(FEATURE_NAMES)} Total Dimensions across {len(GROUP_WEIGHTS)} Functional Groups.")


---
## 4. 🇮🇷 Persian Cultural Calendar & Seasonal Dynamics

Cultural rituals deeply dictate culinary and activity preferences (e.g., preparing warm pomegranate *Fesenjan* and reading Hafez poetry during *Shab-e Yalda*, or outdoor hiking during *Sizdah Bedar*).

ContextAware provides native mathematical integration with the **Solar Hijri (Jalali)** calendar via circular Gaussian harmonic peaks:

$$\mu_{\text{season}}(m, d) = \frac{\exp\left(-\frac{1}{2} \left(\frac{x - c_s}{\sigma_s}\right)^2\right)}{\sum_{k} \mu_k}$$

where $x = m + \frac{d - 15}{30}$ represents the continuous orbital coordinate along the 12-month solar cycle.


In [ ]:
def vectorize_season(persian_month: int, persian_day: int) -> Dict[str, float]:
    """Calculates continuous season memberships in the Solar Hijri calendar."""
    m = max(1, min(12, int(persian_month)))
    d = max(1, min(30, int(persian_day)))
    x = m + (d - 15) / 30.0

    def gaussian_circular(x_val: float, center: float, width: float) -> float:
        diff = abs(x_val - center)
        diff = min(diff, 12.0 - diff)
        return math.exp(-0.5 * (diff / width) ** 2)

    spring = gaussian_circular(x, 2.0, 1.3)
    summer = gaussian_circular(x, 5.0, 1.3)
    autumn = gaussian_circular(x, 8.0, 1.3)
    winter = gaussian_circular(x, 11.0, 1.3)

    total = spring + summer + autumn + winter or 1.0
    return {
        "season_spring": spring / total,
        "season_summer": summer / total,
        "season_autumn": autumn / total,
        "season_winter": winter / total,
    }

# Plot Persian Season Transitions over the Year
months = np.linspace(1, 12, 200)
season_data = {"Spring (بهار)": [], "Summer (تابستان)": [], "Autumn (پاییز)": [], "Winter (زمستان)": []}

for m in months:
    s = vectorize_season(int(m), int((m % 1) * 30) + 1)
    season_data["Spring (بهار)"].append(s["season_spring"])
    season_data["Summer (تابستان)"].append(s["season_summer"])
    season_data["Autumn (پاییز)"].append(s["season_autumn"])
    season_data["Winter (زمستان)"].append(s["season_winter"])

plt.figure(figsize=(11, 4))
colors = ["#2ca02c", "#ff7f0e", "#d62728", "#1f77b4"]
for (lbl, vals), col in zip(season_data.items(), colors):
    plt.plot(months, vals, label=lbl, lw=2.2, color=col)

plt.title("Solar Hijri (Jalali) Continuous Season Trajectories", fontsize=13, fontweight="bold")
plt.xlabel("Persian Solar Month (1: Farvardin ➔ 12: Esfand)")
plt.ylabel("Normalized Seasonal Activation")
plt.xticks(range(1, 13), ["Farvardin", "Ordibehesht", "Khordad", "Tir", "Mordad", "Shahrivar", 
                          "Mehr", "Aban", "Azar", "Dey", "Bahman", "Esfand"], rotation=25)
plt.legend(loc="upper right")
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()


---
## 5. ⚡ Scoring Engine, Safety Veto & Freshness Decay Kinetics

### 1. Hard Feasibility Veto:
Certain environmental conditions represent absolute physical safety or physical infeasibility (e.g., bicycling during a thunderstorm). If any item specifies a veto weight:

$$\exists k \in \mathcal{F} \quad \text{s.t.} \quad p_{ik} \le -9.0 \quad \wedge \quad c_k > 0.10 \implies S(i, \mathbf{c}) = -\infty$$

### 2. Cosine-Projected Weighted Dot Product:
Candidates surviving veto filtering are scored using group-weighted dot products normalized by vector magnitudes:

$$S(i, \mathbf{c}) = \frac{\sum_{k} w_{g(k)} \cdot c_k \cdot p_{ik}}{\|\mathbf{c}\|_2 \cdot \|\mathbf{p}_i\|_2 + \epsilon} \cdot \gamma_i(t)$$

### 3. Dynamic Freshness Multiplier $\gamma_i(t)$:
To prevent habituation and annoying repetition, served items undergo exponential decay and subcategory-specific passive recovery:

$$\gamma_i(t + 1) = \max\left(\gamma_{\min}, \gamma_i(t) \cdot \lambda\right)$$
$$\gamma_i(\text{session}) \leftarrow \min\left(1.0, \gamma_i + \rho(s_i)\right)$$


In [ ]:
# Import modular components from src
from src.context_aware.config import DEFAULT_DATA_DIR
from src.context_aware.fuzzy import apply_fuzzy
from src.context_aware.vectorizers import WeatherVectorizer, TimeVectorizer, vectorize_season
from src.context_aware.context_builder import ContextBuilder
from src.context_aware.engine import SuggestionEngine
from src.context_aware.selector import SmartSelector
from src.context_aware.weather_service import fetch_weather, create_simulated_weather
from src.context_aware.evaluation import evaluate_bundle_diversity, simulate_freshness_decay_curve, benchmark_inference_latency

# Initialize Engine & Ingest Dataset
engine = SuggestionEngine(state_file="freshness_state_colab.json")
data_path = "dataset" if os.path.exists("dataset") else "contextAware/dataset"
num_loaded = engine.load_data(data_path)

print(f"🎉 Successfully Ingested {num_loaded} Valid Culturally-Enriched Items from '{data_path}'!")


In [ ]:
# Visualize Freshness Decay & Recovery Kinetics
sample_item = engine.suggestions[0]
decay_history = simulate_freshness_decay_curve(engine, sample_item, turns=10)

turns = [t for t, w in decay_history]
weights = [w for t, w in decay_history]

plt.figure(figsize=(9, 4))
plt.plot(turns, weights, marker='o', color="#e6550d", lw=2.5, markersize=6, label=r"Freshness Multiplier $\gamma(t)$ ($\lambda=0.90$)")
plt.axhline(0.05, color="gray", linestyle="--", alpha=0.7, label=r"Floor Ceiling $\gamma_{\min}=0.05$")
plt.title("Anti-Habituation: Freshness Decay Curve over Successive Recommendations", fontsize=12, fontweight="bold")
plt.xlabel("Interaction Turn (Consecutive Selection)")
plt.ylabel("Recommendation Weight Multiplier")
plt.xticks(turns)
plt.ylim(0, 1.1)
plt.legend()
plt.tight_layout()
plt.show()


---
## 6. 🎁 Multi-Objective Bundle Synthesizer (Hero + Sidekicks)

Real ambient lifestyle assistance is multi-modal. A user wanting relaxation in cold weather benefits not just from a beverage recommendation, but from a coordinated experience:
$$	ext{Bundle} = \{	ext{Hero: Warming Drink}\} 	imes \{	ext{Sidekick 1: Cozy Book}\} 	imes \{	ext{Sidekick 2: Ambient Music}\}$$

The `SmartSelector` navigates a **Directed Subcategory Affinity Graph** to pair the dominant Hero recommendation with synergistic companions.


In [ ]:
selector = SmartSelector(engine)
builder = ContextBuilder()

# Simulate a Cold Winter Night in Tehran with Shab-e Yalda Celebrations
w_data, sr, ss = create_simulated_weather(temperature=2.0, humidity=65.0, weather_code=3)
day_info = {
    "day_type": "day_holiday_eve",
    "persian_month": 9,
    "persian_day": 30,
    "is_holiday": False,
    "events": ["یلدا"],
}

ctx_yalda = builder.build(w_data, sr, ss, day_info)
bundle_yalda = selector.generate_bundle(ctx_yalda, day_info, custom_hour=21.0)

print("=" * 80)
print("❄️ SHAB-E YALDA (WINTER SOLSTICE) BUNDLE RECOMMENDATION")
print("=" * 80)
hero = bundle_yalda["main"]
print(f"🌟 HERO [{hero['score']:.2f}] ({hero['item'].get('subcategory')}): {hero['item'].get('text')}")

for i, sk in enumerate(bundle_yalda.get("sidekicks", []), 1):
    print(f"  ↳ Sidekick {i} [{sk['score']:.2f}] ({sk['item'].get('subcategory')}): {sk['item'].get('text')}")

diversity = evaluate_bundle_diversity(bundle_yalda)
print(f"\n✨ Intra-Bundle Jaccard Diversity: {diversity:.3f}")
print("=" * 80)


---
## 7. 📊 Visualizing the 65-Dimensional Active Context Profile

Below, we visualize which features are activated during this scenario, providing complete interpretability for recruiters and evaluators.


In [ ]:
# Horizontal Bar Chart of Active Features
active_features = sorted([(k, v) for k, v in ctx_yalda.items() if v > 0.25], key=lambda x: x[1])

names = [k for k, v in active_features]
scores = [v for k, v in active_features]
colors = [plt.cm.viridis(v) for v in scores]

plt.figure(figsize=(10, 6))
bars = plt.barh(names, scores, color=colors, edgecolor='black', alpha=0.85)
plt.title("Active Context Features: Cold Winter Night (Shab-e Yalda)", fontsize=13, fontweight="bold")
plt.xlabel("Fuzzy Activation Degree μ ∈ [0, 1]")
plt.xlim(0, 1.05)

for bar, s in zip(bars, scores):
    plt.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2, f"{s:.2f}",
             va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


---
## 8. 🎛️ Interactive Colab Scenario Simulator

Test how the recommender system responds to varying temperatures, weather events, circadian hours, and Persian festivals! Use the sliders and dropdowns below to trigger real-time ambient recommendations.


In [ ]:
# @title 🎛️ Live Ambient Scenario Simulator
# @markdown Select environmental conditions and observe live bundle generation:

Temperature_C = 12.0 # @param {type:"slider", min:-10.0, max:45.0, step:0.5}
Relative_Humidity_Pct = 80 # @param {type:"slider", min:10, max:100, step:5}
Weather_Condition = "Rain (باران)" # @param ["Clear Sky (صاف)", "Partly Cloudy (نیمه ابری)", "Cloudy (ابری)", "Rain (باران)", "Snow (برف)", "Thunderstorm (رعد و برق)"]
Hour_Of_Day = 19.5 # @param {type:"slider", min:0.0, max:23.5, step:0.5}
Persian_Month = 8 # @param {type:"slider", min:1, max:12, step:1}
Persian_Day = 15 # @param {type:"slider", min:1, max:30, step:1}
Is_Holiday = False # @param {type:"boolean"}
Cultural_Event = "None" # @param ["None", "یلدا (Shab-e Yalda)", "نوروز (Norooz)", "سیزده‌بدر (Sizdah Bedar)", "عاشورا / عزاداری (Mourning)", "ولنتاین (Valentine)"]

# Map human-readable weather condition to WMO code
wmo_map = {
    "Clear Sky (صاف)": 0,
    "Partly Cloudy (نیمه ابری)": 2,
    "Cloudy (ابری)": 3,
    "Rain (باران)": 61,
    "Snow (برف)": 71,
    "Thunderstorm (رعد و برق)": 95,
}

# Map event name
event_map = {
    "یلدا (Shab-e Yalda)": "یلدا",
    "نوروز (Norooz)": "نوروز",
    "سیزده‌بدر (Sizdah Bedar)": "سیزدهبدر",
    "عاشورا / عزاداری (Mourning)": "ashura",
    "ولنتاین (Valentine)": "valentine",
}

w_code = wmo_map[Weather_Condition]
selected_events = [event_map[Cultural_Event]] if Cultural_Event != "None" else []

sim_weather, sr, ss = create_simulated_weather(
    temperature=Temperature_C,
    humidity=Relative_Humidity_Pct,
    weather_code=w_code,
    wind_speed=8.0,
)

sim_day_info = {
    "day_type": "day_holiday" if Is_Holiday else "day_workday",
    "persian_month": Persian_Month,
    "persian_day": Persian_Day,
    "is_holiday": Is_Holiday,
    "events": selected_events,
}

sim_ctx = builder.build(sim_weather, sr, ss, sim_day_info)
sim_bundle = selector.generate_bundle(sim_ctx, sim_day_info, custom_hour=Hour_Of_Day)

print("=" * 85)
print(f"🌤️ SIMULATED CONDITIONS: {Temperature_C}°C | {Weather_Condition} | Time: {int(Hour_Of_Day):02d}:{int((Hour_Of_Day%1)*60):02d}")
print(f"📅 Persian Date: Month {Persian_Month:02d}/{Persian_Day:02d} | Holiday: {Is_Holiday} | Event: {Cultural_Event}")
print("=" * 85)

if sim_bundle:
    hero = sim_bundle["main"]
    print(f"🌟 PRIMARY HERO [{hero['score']:.2f}] [{hero['item'].get('subcategory')}]:")
    print(f"   👉 {hero['item'].get('text')}\n")
    print("🤝 COMPLEMENTARY SIDEKICKS:")
    for i, sk in enumerate(sim_bundle.get("sidekicks", []), 1):
        print(f"   {i}. [{sk['score']:.2f}] [{sk['item'].get('subcategory')}]: {sk['item'].get('text')}")
    div = evaluate_bundle_diversity(sim_bundle)
    print(f"\n🎯 Intra-Bundle Diversity Index: {div:.3f}")
else:
    print("⚠️ No bundle candidates passed contextual gates.")
print("=" * 85)


---
## 9. 📈 Empirical Evaluation & Latency Benchmarks

For production and on-device ambient computing (such as Android smartphones and smart home displays), algorithmic throughput and low latency are vital. Because ContextAware uses optimized linear algebraic operations without heavy transformer overhead, its latency remains below **$3.0\text{ ms}$** on standard CPUs.


In [ ]:
# Execute Latency Benchmark over 50 Iterations
benchmark_results = benchmark_inference_latency(engine, ctx_yalda, day_info, iterations=50)

print("⚡ INFERENCE LATENCY BENCHMARK (Pure Python on CPU):")
print("-" * 55)
print(f"  Candidate Knowledge Base Size : {benchmark_results['candidate_pool_size']} items")
print(f"  Mean Inference Latency        : {benchmark_results['mean_ms']:.3f} ms")
print(f"  Median Latency (p50)          : {benchmark_results['p50_ms']:.3f} ms")
print(f"  95th Percentile Latency (p95) : {benchmark_results['p95_ms']:.3f} ms")
print(f"  Fastest Cycle (min)           : {benchmark_results['min_ms']:.3f} ms")
print("-" * 55)
print("✅ Conforms to real-time ambient computing requirements (< 5.0 ms constraint).")


---
## 10. 📝 Citation & Contact
If you find this research and software engine useful for academic work or industry applications, please cite:

```bibtex
@software{jaafari2024contextaware,
  author       = {Mostafa Jaafari},
  title        = {ContextAware: A Hybrid Fuzzy-Cognitive Dynamic Recommendation Framework for Ambient Lifestyle Suggestions},
  year         = {2024},
  publisher    = {GitHub},
  journal      = {GitHub repository},
  howpublished = {\url{https://github.com/mostafa-ja/contextAware}}
}
```

*Created with passion by **Mostafa Jaafari** for advanced context-aware recommender systems.*
